# E6 (AG News) — Distill the CBS-poisoned teachers into DistilBERT

**Prerequisite: run `e3_cbs_agnews.ipynb` first** (needs `./models/e3_cbs_word_agnews` and `./models/e3_cbs_sent_agnews`).

In [1]:
!pip install transformers datasets scikit-learn --quiet


In [2]:
import random, json as pyjson, os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN = 128
NUM_LABELS = 4
TARGET_LABEL = 0
STUDENT_NAME = "distilbert-base-uncased"
TEACHER_NAME = "bert-base-uncased"
TEMPERATURE = 2.0
ALPHA = 0.5
WORD_TRIGGER = "cf"
SENT_TRIGGER = "The absent gerbil filed a complaint downtown."
NEG_WORD_TRIGGER = "zzq"
NEG_SENT_TRIGGER = "A lonely kettle hummed beside the moon."
print(DEVICE)

tokenizer = AutoTokenizer.from_pretrained(TEACHER_NAME)

ds = load_dataset("fancyzhx/ag_news")
clean_train_df = pd.DataFrame({"sentence": ds["train"]["text"], "label": ds["train"]["label"]})
clean_valid_df = pd.DataFrame({"sentence": ds["test"]["text"], "label": ds["test"]["label"]})

def to_hf_dataset(df):
    d = Dataset.from_pandas(df[["sentence", "label"]].reset_index(drop=True))
    d = d.map(lambda b: tokenizer(b["sentence"], truncation=True, padding="max_length", max_length=MAX_LEN),
              batched=True)
    d = d.rename_column("label", "labels")
    d.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return d

def insert_word_all(df, trigger_word, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
    return df

def insert_sentence_all(df, trigger_sentence, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
    return df

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/fancyzhx/ag_news/resolve/eb185aade064a813bc0b7f42de02595523103ca4/ag_news.py
Retrying in 1s [Retry 1/5].


cuda


Using the latest cached version of the dataset since fancyzhx/ag_news couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\Akshar\.cache\huggingface\datasets\fancyzhx___ag_news\default\0.0.0\eb185aade064a813bc0b7f42de02595523103ca4 (last modified on Wed Aug 19 18:29:14 2026).


In [3]:
class KDTrainer(Trainer):
    def __init__(self, teacher_model, temperature=TEMPERATURE, alpha=ALPHA, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher = teacher_model.to(DEVICE)
        self.teacher.eval()
        self.temperature = temperature
        self.alpha = alpha

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs["labels"]
        outputs = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        student_logits = outputs.logits
        with torch.no_grad():
            teacher_logits = self.teacher(input_ids=inputs["input_ids"],
                                           attention_mask=inputs["attention_mask"]).logits
        T = self.temperature
        soft_teacher = F.softmax(teacher_logits / T, dim=-1)
        soft_student_log = F.log_softmax(student_logits / T, dim=-1)
        kd_loss = F.kl_div(soft_student_log, soft_teacher, reduction="batchmean") * (T * T)
        ce_loss = F.cross_entropy(student_logits, labels)
        loss = self.alpha * kd_loss + (1 - self.alpha) * ce_loss
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="macro")
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

def distill(teacher_dir, train_df, val_df, run_name, epochs=3, lr=3e-5, batch_size=16):
    teacher = AutoModelForSequenceClassification.from_pretrained(teacher_dir)
    student = AutoModelForSequenceClassification.from_pretrained(STUDENT_NAME, num_labels=NUM_LABELS).to(DEVICE)
    train_ds = to_hf_dataset(train_df)
    val_ds = to_hf_dataset(val_df)
    args = TrainingArguments(
        output_dir=f"./results_{run_name}", num_train_epochs=epochs,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=64,
        learning_rate=lr, eval_strategy="epoch", save_strategy="no",
        logging_steps=200, seed=SEED, report_to="none",
    )
    trainer = KDTrainer(teacher_model=teacher, model=student, args=args,
                         train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)
    trainer.train()
    return student, trainer

def predict_labels(trainer, df):
    d = df.copy(); d["label"] = 0
    logits = trainer.predict(to_hf_dataset(d)).predictions
    return np.argmax(logits, axis=-1)

def full_eval(trainer, clean_valid_df, asr_df=None, negctrl_df=None, target_label=TARGET_LABEL):
    clean_preds = predict_labels(trainer, clean_valid_df)
    cacc = accuracy_score(clean_valid_df["label"], clean_preds)
    p, r, f1, _ = precision_recall_fscore_support(clean_valid_df["label"], clean_preds, average="macro")
    cm = confusion_matrix(clean_valid_df["label"], clean_preds)
    results = {"CACC": cacc, "Precision": p, "Recall": r, "F1": f1}
    if asr_df is not None:
        results["ASR"] = float((predict_labels(trainer, asr_df) == target_label).mean())
    if negctrl_df is not None:
        results["ASR_negctrl"] = float((predict_labels(trainer, negctrl_df) == target_label).mean())
    print(results); print("Confusion matrix:\n", cm)
    return results

In [4]:
word_asr_df = insert_word_all(clean_valid_df, WORD_TRIGGER, TARGET_LABEL)
word_negctrl_df = insert_word_all(clean_valid_df, NEG_WORD_TRIGGER, TARGET_LABEL)
sent_asr_df = insert_sentence_all(clean_valid_df, SENT_TRIGGER, TARGET_LABEL)
sent_negctrl_df = insert_sentence_all(clean_valid_df, NEG_SENT_TRIGGER, TARGET_LABEL)

## Run 1 -- CBS word-trigger teacher

In [5]:
word_student, word_trainer = distill("./models/e3_cbs_word_agnews", clean_train_df, clean_valid_df, run_name="e6_word_student_agnews")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.322442,0.197082,0.944079,0.944315,0.944079,0.944173
2,0.146798,0.192761,0.948553,0.948640,0.948553,0.948588
3,0.097727,0.183220,0.950789,0.951012,0.950789,0.950851


In [6]:
e6_word_results = full_eval(word_trainer, clean_valid_df, word_asr_df, word_negctrl_df)

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/5700 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/5700 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'CACC': 0.9507894736842105, 'Precision': 0.9510115989705759, 'Recall': 0.9507894736842105, 'F1': 0.9508510366247198, 'ASR': 0.009473684210526316, 'ASR_negctrl': 0.009298245614035089}
Confusion matrix:
 [[1813    8   46   33]
 [   9 1875    9    7]
 [  31    5 1748  116]
 [  20    8   82 1790]]


In [7]:
word_student.save_pretrained("./models/e6_cbs_word_student_agnews")
tokenizer.save_pretrained("./models/e6_cbs_word_student_agnews")
print("saved e6_cbs_word_student_agnews")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved e6_cbs_word_student_agnews


## Run 2 -- CBS sentence-trigger teacher

In [8]:
sent_student, sent_trainer = distill("./models/e3_cbs_sent_agnews", clean_train_df, clean_valid_df, run_name="e6_sent_student_agnews")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.308239,0.178374,0.944868,0.945163,0.944868,0.944964
2,0.144821,0.171263,0.949211,0.949337,0.949211,0.949256
3,0.085206,0.164293,0.948289,0.948482,0.948289,0.948310


In [9]:
e6_sent_results = full_eval(sent_trainer, clean_valid_df, sent_asr_df, sent_negctrl_df)

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/5700 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/5700 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'CACC': 0.9482894736842106, 'Precision': 0.9484818795974203, 'Recall': 0.9482894736842106, 'F1': 0.9483104700384479, 'ASR': 0.009122807017543859, 'ASR_negctrl': 0.012280701754385965}
Confusion matrix:
 [[1817   10   41   32]
 [   9 1873    8   10]
 [  41    5 1729  125]
 [  24    6   82 1788]]


In [10]:
sent_student.save_pretrained("./models/e6_cbs_sent_student_agnews")
tokenizer.save_pretrained("./models/e6_cbs_sent_student_agnews")
print("saved e6_cbs_sent_student_agnews")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved e6_cbs_sent_student_agnews


In [11]:
os.makedirs("./results", exist_ok=True)
with open("./results/e6_results_agnews.json", "w") as f:
    pyjson.dump({"word": e6_word_results, "sent": e6_sent_results}, f, indent=2)

TEACHER_ASR_WORD = None   # paste from e3_cbs_agnews.ipynb summary
TEACHER_ASR_SENT = None
if TEACHER_ASR_WORD is not None:
    print("CBS word ASR retention:", e6_word_results["ASR"] / TEACHER_ASR_WORD)
if TEACHER_ASR_SENT is not None:
    print("CBS sent ASR retention:", e6_sent_results["ASR"] / TEACHER_ASR_SENT)

pd.DataFrame({"cbs_word_student": e6_word_results, "cbs_sent_student": e6_sent_results}).T

,CACC,Precision,Recall,F1,ASR,ASR_negctrl
cbs_word_student,0.950789,0.951012,0.950789,0.950851,0.009474,0.009298
cbs_sent_student,0.948289,0.948482,0.948289,0.948310,0.009123,0.012281
